# Week1_ex3 - Region Boundary Proximity Check (companion)

**Background**: Hypothesis 1 (adaptive mesh reused from the `magnet_z0 = -1cm` nominal point) was checked manually in the AEDT GUI (Setup1 -> Convergence, per variation) and rejected -- every sampled position converges independently with its own pass count and tetrahedra count, and the nominal point isn't even the best-converged one. See `Week1_ex3_analysis.ipynb` for the full writeup.

**Hypothesis 2**: the magnet is the only real field source in this model (the copper coil carries no excitation and is magnetically transparent). An isolated magnet's self-force should be zero in free space. `Region` approximates free space with a finite padded box, so the non-zero, sign-changing `Force_z` curve may come from the magnet's proximity to this finite boundary rather than from any real physical interaction.

**This notebook, step 1**: before computing any distance-to-boundary numbers, confirm whether `Region`'s bounding box is fixed (sized once when `create_region()` ran, at `magnet_z0 = -1cm`) or gets recomputed per variation as the magnet moves. If it's fixed, the boundary-proximity story is geometrically straightforward. If it moves with the magnet, this hypothesis doesn't hold and needs to be dropped.

**Important**: this notebook temporarily changes the `magnet_z0` design variable to query geometry at different positions. It does **not** call `save_project()` anywhere -- closing AEDT without saving afterward leaves the solved project untouched.

In [1]:
# 1. Imports
import os
import pandas as pd
import ansys.aedt.core

# 2. Suppress screen logging so local paths don't leak into notebook output
ansys.aedt.core.settings.enable_screen_logs = False


In [2]:
# 3. Attach to the already-solved Week1_ex3 project (no re-solve)
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)

# 4. Rebuild the same path the original notebook saved to
proj_name = "Week1_ex3"
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
project_path = os.path.join(base_dir, proj_name, f"{proj_name}.aedt")

# 5. Re-attach Maxwell3d to the existing project and design
# remove_lock=True guards against a stale .aedt.lock from an earlier failed attach
M3D = ansys.aedt.core.Maxwell3d(project=project_path, design="Week1_ex3", remove_lock=True)


PyAEDT INFO: Python version 3.12.13 | packaged by Anaconda, Inc. | (main, Jul  9 2026, 14:26:47) [MSC v.1942 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.15.3.
PyAEDT INFO: Initializing new Desktop session.


In [3]:
# 6. Query Region's bounding box at three different magnet_z0 values
# If the box is identical across all three rows, Region is fixed (not tracking the magnet).
# If it shifts, Region is being recomputed per variation and Hypothesis 2 needs rethinking.
test_points = ["-5cm", "-1cm", "3cm"]
rows = []
for zp in test_points:
    M3D["magnet_z0"] = zp
    bbox = M3D.modeler["Region"].bounding_box
    rows.append({"magnet_z0": zp, "bounding_box": bbox})

# 7. Restore the nominal value so the design is left the way it was found
M3D["magnet_z0"] = "-1cm"

df_region = pd.DataFrame(rows)
df_region


c:\Users\abane\anaconda\envs\pyaedt_env\Lib\site-packages\ansys\aedt\core\visualization\advanced\misc.py:43: UserWarning: The PyVista module is required to run functionalities of ansys.aedt.core.visualization.advanced.misc.
Install with 

pip install pyvista
  warnings.warn(


,magnet_z0,bounding_box
0,-5cm,"[-5.0, -5.0, -8.75, 5.0, 5.0, 6.25]"
1,-1cm,"[-5.0, -5.0, -5.0, 5.0, 5.0, 5.0]"
2,3cm,"[-5.0, -5.0, -6.25, 5.0, 5.0, 8.75]"
